The below model training is the final effect of larger Python process and practice, and multiple failed attempts, as I tried to predict direction first. This turned out to be a bad idea and I finally turned towards volatility predictions, which have clear research studies backup + show autocorrelation and real predictability.

In order to develop proper features I tested some basic concepts like ATR, day of week dependence, candle streaks etc., but at some point I checked research papers

This is just to show off the training process, as the model itself is wrapped and running on Render and called through FastAPI with a scheduled Cloudflare worker (cron).



In [1]:
#imports here

#More rationale and process is shown in python_agentic_learning repo, practice.ipynb,
# as I've tested multiple angles and options (at first I tried to predict direction)

#Some features are math heavy, so I've obviously had to use LLMs or got the funcs from the web.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Data example is visible in xauusd_m5_et_example.csv
m5_df = pd.read_csv('xauusd_m5_et.csv', parse_dates=['et_time'])

# absolute 1-bar returns
m5_df['ret_1bar'] = m5_df['close'].pct_change()
m5_df['abs_ret_1bar'] = m5_df['ret_1bar'].abs()

# atr14 (Wilder ATR, 14-period) - M5 file has no atr14 column, built fresh here
m5_df['prev_close'] = m5_df['close'].shift(1)
m5_df['tr'] = m5_df[['high', 'low', 'prev_close']].apply(
    lambda row: max(row['high'] - row['low'],
                    abs(row['high'] - row['prev_close']),
                    abs(row['low'] - row['prev_close'])), axis=1)
m5_df['atr14'] = m5_df['tr'].ewm(alpha=1/14, adjust=False).mean()

# otf_streak - exact function from Week 9 Day 1 (repo -> python_agentic_learning/practice.ipynb :)), unchanged
from typing import List
def otf_streak(highs: List, lows: List) -> List:
    otf_streaks = [0] * len(highs)
    for i in range(1, len(otf_streaks)):
        if (lows[i] > lows[i-1]) and (highs[i] < highs[i-1]):
            otf_streaks[i] = otf_streaks[i-1]
        elif lows[i] > lows[i-1]:
            otf_streaks[i] = 1 if otf_streaks[i-1] < 0 else otf_streaks[i-1] + 1
        elif highs[i] < highs[i-1]:
            otf_streaks[i] = -1 if otf_streaks[i-1] > 0 else otf_streaks[i-1] - 1
    return otf_streaks
m5_df['otf_streak'] = otf_streak(m5_df['high'].tolist(), m5_df['low'].tolist())

# atr_trend (1 if atr rising over last 5 bars, else 0 - same def as Week 8)
m5_df['atr_trend'] = (m5_df['atr14'] - m5_df['atr14'].shift(5) > 0).astype(int)

# efficiency_ratio_10, sjv_10, center_of_mass, wtc_10 - same window-10 defs as Week 8 Day 3
m5_df['net_displacement_10'] = m5_df['ret_1bar'].rolling(10).sum().abs()
m5_df['realized_power_10'] = m5_df['abs_ret_1bar'].rolling(10).sum()
m5_df['efficiency_ratio_10'] = m5_df['net_displacement_10'] / m5_df['realized_power_10']

m5_df['sq_ret_pos_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
m5_df['sq_ret_neg_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
m5_df['sjv_10'] = (m5_df['sq_ret_pos_10'] - m5_df['sq_ret_neg_10']) / (m5_df['sq_ret_pos_10'] + m5_df['sq_ret_neg_10'])

def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0
    raw_com = np.sum(positions * abs_returns_window) / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius
m5_df['center_of_mass'] = m5_df['abs_ret_1bar'].rolling(10).apply(center_of_mass, raw=True)

m5_df['wtc_10'] = np.log(m5_df['high'] / m5_df['low']).rolling(10).sum() / m5_df['realized_power_10']

In [2]:
m5_df['is_obs'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute < 15)
m5_df['is_pred'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute >= 15)

print(m5_df.columns)
observation_window = m5_df.groupby(['is_obs', 'trade_date']).agg(
    realized_power_observation = ('abs_ret_1bar', 'sum'),
    atr14_mean_observation = ('atr14', 'mean'),
    otf_streak_observation = ('otf_streak', 'last'),
    sjv_10_observation = ('sjv_10', 'mean'),
    efficiency_ratio_10_observation = ('efficiency_ratio_10', 'mean'),
    wtc_10_observation = ('wtc_10', 'mean'),
    center_of_mass_mean_observation = ('center_of_mass', 'mean'),
    atr_trend_observation = ('atr_trend', 'last'),
).reset_index()

predicted_window = m5_df.groupby(['is_pred', 'trade_date']).agg(
    realized_power_predicted = ('abs_ret_1bar', 'sum'),
    atr14_mean_predicted = ('atr14', 'mean'),
    otf_streak_predicted = ('otf_streak', 'last'),
    sjv_10_predicted = ('sjv_10', 'mean'),
    efficiency_ratio_10_predicted = ('efficiency_ratio_10', 'mean'),
    wtc_10_predicted = ('wtc_10', 'mean'),
    center_of_mass_mean_predicted = ('center_of_mass', 'mean'),
    atr_trend_predicted = ('atr_trend', 'last'),
).reset_index()

observation_window = observation_window[observation_window['is_obs'] == True]
predicted_window = predicted_window[predicted_window['is_pred'] == True]



merged_df = observation_window.merge(predicted_window, how = 'inner', on = 'trade_date')
merged_df = merged_df.dropna()


merged_df['target_log_ratio'] = np.log(merged_df['realized_power_predicted'] / merged_df['realized_power_observation'])
merged_df['day_of_week'] = pd.to_datetime(merged_df['trade_date']).dt.day_of_week

merged_df.head()




Index(['et_time', 'open', 'high', 'low', 'close', 'volume', 'day_of_week',
       'trade_date', 'ret_1bar', 'abs_ret_1bar', 'prev_close', 'tr', 'atr14',
       'otf_streak', 'atr_trend', 'net_displacement_10', 'realized_power_10',
       'efficiency_ratio_10', 'sq_ret_pos_10', 'sq_ret_neg_10', 'sjv_10',
       'center_of_mass', 'wtc_10', 'is_obs', 'is_pred'],
      dtype='str')


,is_obs,trade_date,realized_power_observation,atr14_mean_observation,otf_streak_observation,sjv_10_observation,efficiency_ratio_10_observation,wtc_10_observation,center_of_mass_mean_observation,atr_trend_observation,...,realized_power_predicted,atr14_mean_predicted,otf_streak_predicted,sjv_10_predicted,efficiency_ratio_10_predicted,wtc_10_predicted,center_of_mass_mean_predicted,atr_trend_predicted,target_log_ratio,day_of_week
0,True,2021-01-04,0.001573,2.141354,1,0.638500,0.353078,2.025449,-0.187964,0,...,0.006183,2.325681,0,0.167199,0.165138,2.027781,0.063296,1,1.368961,0
1,True,2021-01-05,0.002255,2.174545,1,0.124336,0.105011,1.663593,0.082387,1,...,0.005394,2.470881,1,-0.179032,0.260950,2.009797,-0.047712,1,0.872319,1
2,True,2021-01-06,0.002614,3.537418,1,-0.359701,0.446023,1.955187,0.094091,0,...,0.011761,3.663594,2,-0.116528,0.119847,1.686894,0.088945,1,1.504021,2
3,True,2021-01-07,0.001501,2.137776,-1,-0.459122,0.187134,1.925579,0.132577,0,...,0.012448,2.661309,-1,-0.294899,0.126671,1.769492,0.130867,1,2.115136,3
4,True,2021-01-08,0.002423,3.753778,0,0.247682,0.189025,1.756307,-0.074038,1,...,0.015592,3.761374,-7,-0.249180,0.508533,1.771523,0.009624,1,1.861697,4


In [3]:
merged_df['day_of_week'] = pd.to_datetime(merged_df['trade_date']).dt.day_of_week

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]


baseline_mae = np.full(len(test), train['target_log_ratio'].mean())
baseline_mae = mean_absolute_error(baseline_mae, test['target_log_ratio'])
print(baseline_mae) #0.494

scorer_mae = make_scorer(mean_absolute_error)
features = [
    "realized_power_observation",
    "atr14_mean_observation",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]

param_grid = {
     'max_depth': [3, 4, 5, 6],
     'min_samples_leaf': [5, 10, 20, 30]
}
searcher = RandomizedSearchCV(
     estimator = RandomForestRegressor(),
     param_distributions = param_grid,
     n_iter = 20,
     cv = TimeSeriesSplit(n_splits = 5),
     scoring = scorer_mae,
     error_score = 'raise',
     random_state = 42
     )


searcher.fit(train[features], train['target_log_ratio'])
print(searcher.best_params_)

model = RandomForestRegressor(min_samples_leaf = 30, max_depth = 3)
model.fit(train[features], train['target_log_ratio'])
train_predictions = model.predict(train[features])
test_predictions = model.predict(test[features])

train_mae = mean_absolute_error(train['target_log_ratio'], train_predictions)
train_r2 = r2_score(train['target_log_ratio'], train_predictions)

test_r2 = r2_score(test['target_log_ratio'], test_predictions)
test_mae = mean_absolute_error(test['target_log_ratio'], test_predictions)

print(f'Train MAE: {train_mae}, Test MAE: {test_mae}')
print(f'Train R2: {train_r2}, Test R2: {test_r2}')


imps = [importance for importance in model.feature_importances_]
results = pd.DataFrame(
    {
        'feature': [feature for feature in features],
        'importance': [imp for imp in imps]
    }
).sort_values(by = 'importance', ascending = False)
print(results)



0.4939911690216317


c:\Users\HARDPC\Desktop\AL\projekty\python_agentic_learning\venv\Lib\site-packages\sklearn\model_selection\_search.py:326: UserWarning: The total space of parameters 16 is smaller than n_iter=20. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


{'min_samples_leaf': 30, 'max_depth': 3}
Train MAE: 0.29641476967197633, Test MAE: 0.3944069522210371
Train R2: 0.6223842214042571, Test R2: 0.3835037327050106
                           feature  importance
0       realized_power_observation    0.947044
6  center_of_mass_mean_observation    0.030859
1           atr14_mean_observation    0.020474
5               wtc_10_observation    0.000935
8                      day_of_week    0.000495
7            atr_trend_observation    0.000193
4  efficiency_ratio_10_observation    0.000000
2           otf_streak_observation    0.000000
3               sjv_10_observation    0.000000
